In [ ]:
import ..analysis

In [1]:
import analysis
import numpy as np
import audioflux as af
import librosa

In [2]:
dir = "..\\..\\corpus\\case_study_Airbus_A300B4-622R(F)"
sr=48000
grain_duration = 0.1 # 100 ms
grain_size = int(grain_duration * sr)
analyzer = analysis.AnalyzerObject(dir,sr)
df = analyzer.compute_grain_descriptors(grain_size)

c:\Users\hippo\miniconda3\envs\fieldmusic\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Length y: 539649 Length descr y: 539649
Length y: 539649 Length descr y: 539649
Length y: 539649 Length descr y: 539649
Length y: 539649 Length descr y: 539649
Length y: 539649 Length descr y: 539649
Length y: 539649 Length descr y: 539649
Length y: 539649 Length descr y: 539649
Length y: 539649 Length descr y: 539649
Length y: 539649 Length descr y: 539649


In [3]:
analyzer.save_metadata(df, grain_duration)

Saved to csv to: ..\..\corpus\case_study_Airbus_A300B4-622R(F)\metadata\grain_0.1_s_metadata_2ea5b625.csv


In [4]:
metadata_path = "..\..\corpus\case_study_Airbus_A300B4-622R(F)\metadata\grain_0.1_s_metadata_2ea5b625.csv"
df = analyzer.load_metadata(metadata_path)
_, df_scaled = analyzer.scale_metadata(df, 1) # PowerTransformer
df_scaled

,id,sr,index,size,centroid,flux,rolloff,flatness,spread,skewness,kurtosis,crest,rms
0,b1b31a55e,48000,0,4800,-0.114192,-0.547637,-0.196947,782.541579,-0.080429,0.883330,1.463318,-0.118831,-1.106806
1,7d3c5cc08,48000,4800,4800,-1.147512,-0.350130,-0.916475,-0.267566,-0.783510,1.630599,2.712338,0.805256,-0.536816
2,d49e2e144,48000,9600,4800,-0.994949,-0.251744,-0.811012,0.559071,-0.727783,1.448279,2.408794,0.467565,-0.740483
3,6181a478f,48000,14400,4800,-1.172703,-0.489718,-0.948684,-0.768082,-0.863148,1.528350,2.563120,1.031289,-0.554498
4,769602c0e,48000,19200,4800,-1.619483,1.684753,-1.669179,-0.224110,-1.432280,4.244213,11.844074,1.766379,1.122534
...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,8ce6d6ec5,48000,513600,4800,-1.187902,0.244138,-0.836003,0.641251,-0.688965,1.714121,2.572854,0.892960,-0.174704
108,1d0f52ea3,48000,518400,4800,-1.478226,0.906397,-1.212528,0.765104,-1.038177,2.571643,4.664217,1.660206,0.645608
109,7607a9849,48000,523200,4800,-1.364259,0.205015,-1.067416,0.162507,-0.925985,2.169059,3.641942,0.953171,-0.006586
110,e32f8b74f,48000,528000,4800,-1.397528,-0.105955,-1.168332,-0.035620,-1.004420,2.418707,4.377033,1.436175,-0.054176


In [5]:
descriptor_list = ["centroid", "flux", "rolloff", "flatness", "spread", "skewness", "kurtosis", "crest", "rms"]
# df_scaled = df_scaled[descriptor_list]
analysis.get_grain_distr_histograms(dir, df_scaled, descriptor_list, grain_duration)

In [6]:
from algorithms import MarkovGranulizer
import writing

In [7]:
import numpy as np
def rev_exp(size, decay_r=1):
    t = np.linspace(0,1,size)
    window = np.exp(-t/decay_r)
    normalized_window = (window - window[-1])/(window[0]-window[-1])
    return normalized_window
rev_exp(10,1)

def normalize_output(data):
    data_float = data.astype(np.float32)
    peak = np.max(np.abs(data_float))
    if peak == 0:
        normalized = data_float
    else:
        normalized = data_float / peak
    return normalized

In [ ]:
df = analyzer.load_metadata(metadata_path)
_, df_scaled = analyzer.scale_metadata(df, 1) # PowerTransformer
df_scaled


x = "spread"
y = "rolloff"
features = [x,y]
analysis.show_scatter_plt(df_scaled[x], df_scaled[y], x, y, "Grain distribution")

In [ ]:
n_clusters = 2
kmeans_obj = analyzer.compute_kmeans(df_scaled, n_clusters=n_clusters, features=features)
dict_cluster, _ = analyzer.get_cluster_dict(kmeans_obj.labels_)